# EDA — Feature Engineering & Gentrification Risk

Computes daily aggregates, 30-day pollution trends, the environmental improvement index, current pollution levels, volatility, and runs the K-means risk clustering.

In [ ]:
import sys, logging, warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loaders import GreenSentinelLoader, DKVLoader
from src.processors.feature_engineer import FeatureEngineer
from src.models.gentrification_model import GentrificationRiskModel

In [ ]:
gs = GreenSentinelLoader().run().get_data()
dkv = DKVLoader().run()
transit = dkv.compute_transit_accessibility(gs[['station','latitude','longitude']].drop_duplicates())

fe = FeatureEngineer(gs, transit_data=transit)
features = fe.create_all_features()
daily = fe.daily_data
improvement = fe.improvement_metrics

print('Feature matrix:', features.shape)
print(features[['station','env_improvement_index','current_PM2.5','pm25_volatility','bus_stops_nearby']].to_string(index=False))

## Improvement trend by station

In [ ]:
plt.figure(figsize=(10, 6))
imp_sorted = improvement.sort_values('env_improvement_index')
colors = plt.cm.RdYlGn((imp_sorted['env_improvement_index'] - imp_sorted['env_improvement_index'].min()) / (imp_sorted['env_improvement_index'].max() - imp_sorted['env_improvement_index'].min()))
plt.barh(imp_sorted['station'], imp_sorted['env_improvement_index'], color=colors)
plt.xlabel('Environmental Improvement Index')
plt.title('30-day Improvement Index by Station')
plt.tight_layout(); plt.show()

## Risk clustering

In [ ]:
model = GentrificationRiskModel(features).fit_clustering()
report = model.generate_risk_report()
print(report[['station','risk_category','env_improvement_index','current_PM2.5']].to_string(index=False))
print()
print(report['risk_category'].value_counts().to_string())

In [ ]:
plt.figure(figsize=(9, 7))
for cat, color in {'Emerging Green Zones':'#EF4444','Established Clean Areas':'#10B981','Stable Neighborhoods':'#FBBF24','Challenge Zones':'#F97316'}.items():
    sub = report[report['risk_category'] == cat]
    plt.scatter(sub['env_improvement_index'], sub['current_PM2.5'], s=120, c=color, label=cat, edgecolors='white')
    for _, r in sub.iterrows():
        plt.annotate(r['station'], (r['env_improvement_index'], r['current_PM2.5']), textcoords='offset points', xytext=(6, 6), fontsize=8)
plt.axhline(report['current_PM2.5'].median(), ls='--', c='gray', alpha=0.5)
plt.axvline(report['env_improvement_index'].median(), ls='--', c='gray', alpha=0.5)
plt.xlabel('Environmental Improvement Index')
plt.ylabel('Current PM2.5 (µg/m³)')
plt.title('Gentrification Risk Matrix')
plt.legend()
plt.tight_layout(); plt.show()

## Pollutant correlation

In [ ]:
pm = daily.pivot_table(index=['timestamp','station'], columns='measurement_type', values='value')
core = [c for c in ['PM2.5','PM10','NO2','O3','TVOC','CO'] if c in pm.columns]
plt.figure(figsize=(8, 6))
sns.heatmap(pm[core].corr(), annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation between pollutants (daily means)')
plt.tight_layout(); plt.show()